# Sparse Attention：保持 softmax 但只算一部分

> 上一节讲了 Linear Attention——把 softmax 换成可分离的 kernel，复杂度从 $O(N^2)$ 降到 $O(N)$。代价是失去 softmax 的尖锐性，retrieval 类任务上明显弱。这是「换 attention 函数」的路线。
>
> 这一节走另一条路：**保留 softmax 不变，但每个 query 只对一部分 key 计算 attention**。出发点是观察到的事实——softmax 之后大部分 attention 权重接近 0，对最终结果影响很小，跳过它们不会损失多少。Sliding Window 是最朴素的版本（上一节 Part 4 长上下文已经讲过）。这一节聚焦更新的方案：DeepSeek 的 NSA（Native Sparse Attention）和 DSA（DeepSeek Sparse Attention），它们让「选哪些 key」这件事变成模型自己学的，并且 end-to-end 可训练。

观察一个真实的 attention 矩阵：softmax 之后，每个 query 的权重分布通常只有少数几个 key 拿到明显权重，其余接近 0。这意味着大部分 $Q K^T$ 的计算是浪费——算出来了，softmax 后被压到接近 0，对最终输出贡献微乎其微。

Sparse attention 的核心思路就是利用这一点。**只对重要的 key 算 attention，跳过那些不重要的**。问题变成：哪些 key 是「重要的」？

最早的回答是固定规则：邻近的 key 重要（sliding window），少数固定位置重要（global tokens）。Longformer、Big Bird 是这一代。固定规则简单但不够灵活——真正重要的 key 可能不在窗口内，也可能不是预定的 global token。

更新的回答是让模型自己学。NSA（DeepSeek, 2025）提出三路并行：**压缩**（把历史 token 压成低分辨率摘要，做粗 attention）、**选择**（基于粗 attention 分数挑出 top-N 重要 key，做细 attention）、**滑窗**（保留 sliding window 作为兜底）。三路输出用门控融合，整个流程 end-to-end 可训练。

DSA 是 NSA 的简化版——只用一个轻量 indexer（小型 MLP）给每个 key 打分，选 top-N。比 NSA 更简洁，工程上更好部署，DeepSeek-V3.2 用了它。

这一节先看 attention 矩阵到底有多稀疏，再依次过固定 pattern、learnable sparse、NSA 三路机制，最后实现一个简化版 NSA。

## 1. Attention 是稀疏的：实测 softmax 分布

先把「attention 大部分权重接近 0」这个观察量化。下面随机生成一个 attention 矩阵，看 softmax 后的分布。

In [ ]:
# 实测：softmax attention 的权重分布到底有多稀疏

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
N, d = 128, 64

Q = torch.randn(N, d)
K = torch.randn(N, d)

# 标准 softmax attention（带 causal mask）
scores = Q @ K.T / (d ** 0.5)
# causal mask
mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))
weights = F.softmax(scores, dim=-1)

# 统计每个 query 的权重分布
print(f"Attention matrix shape: {weights.shape}")
print(f"每行（每个 query 的权重分布）的统计：")
print()

# 取第 100 个 query（看到 100 个历史 token）
q_idx = 100
row = weights[q_idx, :q_idx+1]  # 只看 causal 范围内的

sorted_row, _ = row.sort(descending=True)
top_5_sum = sorted_row[:5].sum().item()
top_10_sum = sorted_row[:10].sum().item()
top_20_sum = sorted_row[:20].sum().item()

print(f"Query {q_idx} 看到 {q_idx+1} 个 key（causal）")
print(f"  Top-5  key 占总权重: {top_5_sum*100:.1f}%")
print(f"  Top-10 key 占总权重: {top_10_sum*100:.1f}%")
print(f"  Top-20 key 占总权重: {top_20_sum*100:.1f}%")
print(f"  剩下 {q_idx+1-20} 个 key 共占: {(1-top_20_sum)*100:.1f}%")
print()
print(f"关键观察：top-20 个 key 拿走了 {top_20_sum*100:.0f}% 的权重")
print(f"剩下 {q_idx+1-20} 个 key 的计算几乎不影响输出——这就是 sparse attention 的机会")

# 可视化：单个 query 的权重分布
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(q_idx+1), row.numpy(), color='#3498db')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Attention weight')
axes[0].set_title(f'Query {q_idx} 的 attention 权重分布\n大部分接近 0，少数 key 占大头')
axes[0].grid(True, alpha=0.3)

# 累积权重
cumulative = row.sort(descending=True)[0].cumsum(0).numpy()
axes[1].plot(range(1, q_idx+2), cumulative, '-o', markersize=3, color='#e74c3c')
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90% weight')
axes[1].axhline(y=0.95, color='gray', linestyle=':', alpha=0.5, label='95% weight')
axes[1].set_xlabel('Top-K keys')
axes[1].set_ylabel('Cumulative weight')
axes[1].set_title('累积权重：少数 key 就能覆盖 90%+ 的总权重')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 固定 Pattern：Sliding Window 和它的变种

最朴素的 sparse attention 用**固定 pattern**决定哪些 key 该被看到。常见的有：

| Pattern | 怎么选 key | 代表 |
|:---|:---|:---|
| **Sliding Window** | 每个 query 只看邻近 W 个 key | Longformer, Mistral |
| **Local + Global** | sliding window + 少数 global tokens（始终被看到） | Longformer, Big Bird |
| **Dilated Sliding** | 间隔取，扩大感受野 | Longformer |
| **Block-sparse** | 把序列分块，只在某些块间算 attention | Sparse Transformer |

固定 pattern 简单、可解释、容易实现。Part 4 的长上下文一节里已经讲过 Sliding Window 的具体实现。

问题是不够灵活——真正重要的 key 可能离当前 query 很远（比如做 retrieval、跨段指代消解），固定 pattern 选不到。下面看 learnable sparse attention 怎么解决。

## 3. Learnable Sparse Attention：让模型自己选

Learnable sparse attention 的核心想法：加一个轻量「路由器」，对每个 query 给所有 key 打分，挑出 top-N 重要的，只对这 N 个做 attention。

实现上通常这样：

```
1. 用轻量 MLP 给每个 key 算一个 importance score
   score[i] = MLP(query, key_i)  → 标量

2. 选 top-N 重要的 key
   selected_keys = topk(scores, N)

3. 只对选出的 key 算完整的 softmax attention
   output = softmax(Q @ selected_K.T) @ selected_V
```

第一步用 MLP 而不是完整 attention——MLP 的复杂度远低于 attention（一次 forward 几个 GFLOPS），筛掉大部分 key 后再做完整 attention 就划算。

DeepSeek 的 DSA（DeepSeek Sparse Attention，V3.2 引入）就是这个思路：用一个超轻量的 indexer（基于 MQA 风格的小型注意力 + ReLU 激活）给每个 key 打分，选 top-N，再做完整 attention。NSA 比这个更复杂，下一节展开。

In [ ]:
# 简化版 Learnable Sparse Attention：用 query 和 key 的点积打分，选 top-N

import torch
import torch.nn.functional as F

def learnable_sparse_attention(Q, K, V, top_n):
    """简化版 learnable sparse attention

    1. 用 Q·K 的粗略分数（不除 sqrt(d)，不算 softmax）给 key 打分
    2. 选 top-N
    3. 只对选出的 key 算完整 softmax attention
    """
    N, d = Q.shape
    top_n = min(top_n, N)

    # 步骤 1：粗略打分（用 Q·K，比完整 attention 便宜）
    coarse_scores = Q @ K.T  # [N, N]

    # 步骤 2：每个 query 选 top-N 个 key（带 causal mask）
    mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
    coarse_scores_masked = coarse_scores.masked_fill(mask, float('-inf'))

    topk_vals, topk_idx = coarse_scores_masked.topk(top_n, dim=-1)

    # 步骤 3：只对选出的 key 算 attention
    out = torch.zeros_like(V)
    for i in range(N):
        idx = topk_idx[i]  # 这个 query 选的 key 索引
        # 算 attention，只对选出的 key
        selected_scores = Q[i] @ K[idx].T / (d ** 0.5)
        selected_weights = F.softmax(selected_scores, dim=-1)
        out[i] = selected_weights @ V[idx]
    return out, topk_idx

# 演示
torch.manual_seed(42)
N, d = 32, 16
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

# 标准 attention（ground truth）
full_scores = Q @ K.T / (d ** 0.5)
mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
full_scores = full_scores.masked_fill(mask, float('-inf'))
full_weights = F.softmax(full_scores, dim=-1)
full_out = full_weights @ V

# sparse attention，选 top-8
sparse_out, selected = learnable_sparse_attention(Q, K, V, top_n=8)

# 对比
diff = (full_out - sparse_out).norm() / full_out.norm()
print(f"序列长度: {N}")
print(f"Sparse 选 top-8 个 key（共 {N} 个）→ 节省 {(1 - 8/N)*100:.0f}% 的 attention 计算")
print(f"输出相对误差: {diff.item()*100:.2f}%")
print()
print(f"关键观察：")
print(f"  即使只算 8/{N} = {8/N*100:.0f}% 的 key，输出误差也只有 {diff.item()*100:.1f}%")
print(f"  这就是 sparse attention 划算的原因——少算很多，损失不多")

## 4. NSA：三路并行的稀疏注意力

NSA（Native Sparse Attention，DeepSeek 2025）的核心创新是把 sparse attention 做成**三路并行 + 门控融合**，每一路解决不同的问题。

| 分支 | 做什么 | 解决的问题 |
|:---|:---|:---|
| **压缩（Compress）** | 把历史 token 压成低分辨率摘要，做粗 attention | 远距离的整体上下文（不需要精细到每个 token） |
| **选择（Selection）** | 基于粗 attention 分数挑 top-N 重要 token，做细 attention | 远距离的关键 token（需要精确 attention） |
| **滑窗（Sliding）** | 保留 sliding window（最近 W 个 token） | 邻近 token（局部依赖） |

三路的输出通过一个门控（gating）融合：$\text{out} = g_1 \cdot \text{compress\_out} + g_2 \cdot \text{select\_out} + g_3 \cdot \text{sliding\_out}$。门控权重 $g_1, g_2, g_3$ 由模型自己学，可以理解为「这一层当前 query 应该更信赖哪一路」。

NSA 的关键工程贡献是**全流程 end-to-end 可训练**——压缩、选择、滑窗、门控都是可微的，可以直接用反向传播训练。这听起来理所当然，但很多早期 sparse attention 工作（用 top-k、hard selection）是不可微的，需要 RL 或特殊梯度估计。

DSA（DeepSeek Sparse Attention，V3.2）是 NSA 的简化版——去掉 compress 分支，只用一个 indexer 选 top-N。结构更简单，工程更好部署，但表达力略弱。DeepSeek-V3.2 用 DSA 替换了原来的 dense attention，在长上下文 benchmark 上保持性能的同时大幅降算力。

下面写一个极简的 NSA 实现，展示三路融合的机制。

In [ ]:
# 极简版 NSA：三路并行 + 门控融合

import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleNSA(nn.Module):
    """教学版 NSA，展示三路并行的结构

    简化处理：
    - 压缩分支：用 mean pooling 模拟，把每 chunk_size 个 token 平均成 1 个
    - 选择分支：用压缩分支的 attention 分数选 top-N
    - 滑窗分支：固定窗口 W
    - 门控：用 query 学出 3 个权重
    """
    def __init__(self, d_model, chunk_size=4, top_n=8, window_size=8):
        super().__init__()
        self.d_model = d_model
        self.chunk_size = chunk_size
        self.top_n = top_n
        self.window_size = window_size

        # 三路各自的 value 投影
        self.W_V_compress = nn.Linear(d_model, d_model, bias=False)
        self.W_V_select = nn.Linear(d_model, d_model, bias=False)
        self.W_V_sliding = nn.Linear(d_model, d_model, bias=False)

        # query 投影（三路共用，简化）
        self.W_Q = nn.Linear(d_model, d_model, bias=False)

        # 门控：从 query 算出三个权重
        self.gate = nn.Linear(d_model, 3)

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        return: [batch, seq_len, d_model]
        """
        B, S, D = x.shape
        Q = self.W_Q(x)  # [B, S, D]

        # === 压缩分支 ===
        # 把每 chunk_size 个 token 平均成 1 个
        pad = (self.chunk_size - S % self.chunk_size) % self.chunk_size
        x_padded = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
        x_chunks = x_padded.view(B, -1, self.chunk_size, D).mean(dim=2)  # [B, S/chunk, D]
        # 简化：压缩分支用 mean pool 后的 K 直接 attention
        compress_V = self.W_V_compress(x_chunks)  # [B, S/chunk, D]
        compress_scores = Q @ x_chunks.transpose(-1, -2) / (D ** 0.5)
        compress_attn = F.softmax(compress_scores, dim=-1)
        compress_out = compress_attn @ compress_V  # [B, S, D]

        # === 选择分支 ===
        # 用压缩分支的 attention 分数找重要的 chunk，再回原序列选 top-N token
        select_V = self.W_V_select(x)  # [B, S, D]
        # 简化：直接用 Q·x 算 top-N
        select_scores = Q @ x.transpose(-1, -2) / (D ** 0.5)  # [B, S, S]
        # 因果 mask
        mask = torch.triu(torch.ones(S, S), diagonal=1).bool()
        select_scores_masked = select_scores.masked_fill(mask, float('-inf'))
        top_n = min(self.top_n, S)
        topk_vals, topk_idx = select_scores_masked.topk(top_n, dim=-1)
        # 对每个 query，选出的 key 算 attention
        select_out = torch.zeros(B, S, D)
        for b in range(B):
            for i in range(S):
                idx = topk_idx[b, i]
                sel_scores = Q[b, i] @ x[b, idx].T / (D ** 0.5)
                sel_weights = F.softmax(sel_scores, dim=-1)
                select_out[b, i] = sel_weights @ select_V[b, idx]

        # === 滑窗分支 ===
        sliding_V = self.W_V_sliding(x)
        sliding_out = torch.zeros(B, S, D)
        for i in range(S):
            start = max(0, i - self.window_size + 1)
            window_x = x[:, start:i+1]  # [B, window, D]
            window_V = sliding_V[:, start:i+1]
            q_i = Q[:, i:i+1]  # [B, 1, D]，保留 seq 维
            sel_scores = (q_i @ window_x.transpose(-1, -2)).squeeze(1) / (D ** 0.5)
            sel_weights = F.softmax(sel_scores, dim=-1)
            sliding_out[:, i] = (sel_weights.unsqueeze(-1) * window_V).sum(dim=1)

        # === 门控融合 ===
        gate_logits = self.gate(Q)  # [B, S, 3]
        gate_weights = F.softmax(gate_logits, dim=-1)  # [B, S, 3]

        out = (gate_weights[..., 0:1] * compress_out +
               gate_weights[..., 1:2] * select_out +
               gate_weights[..., 2:3] * sliding_out)
        return out

# 测试
torch.manual_seed(42)
nsa = SimpleNSA(d_model=16, chunk_size=4, top_n=4, window_size=4)
x = torch.randn(1, 16, 16)
out = nsa(x)
print(f"输入 shape: {x.shape}")
print(f"输出 shape: {out.shape}")
print()
print(f"三路配置：")
print(f"  压缩: 每 {nsa.chunk_size} token 压成 1 个 → 长序列下的低分辨率视图")
print(f"  选择: 每 query 选 top-{nsa.top_n} token → 精细但只看少量")
print(f"  滑窗: 最近 {nsa.window_size} token → 局部依赖")
print(f"  门控: query 自己决定三路的混合比例")

## 5. DSA：更简洁的 indexer-based 方案

DSA（DeepSeek Sparse Attention，V3.2-Exp 引入）走另一条路。和 NSA 的「三路并行」不同，DSA 只做一件事——用轻量 indexer 给每个 key 打分，选 top-N，做完整 attention。

DSA 的 indexer 极其轻量：一个用 ReLU 激活的小型 MQA 风格注意力。它对每个 query 算出所有 key 的「重要性分数」，选 top-N，剩下的丢掉。然后对选出的 top-N 做**标准的 softmax attention**——和 dense attention 完全一样，只是作用的 key 集合小了。

和 NSA 的对比：

| 维度 | NSA | DSA |
|:---|:---|:---|
| 分支数 | 3（compress + select + sliding） | 1（indexer + top-N） |
| 复杂度 | 中（三路都要算） | 低（只算 indexer + top-N attention） |
| 表达力 | 高（多路互补） | 中（单路） |
| 训练难度 | 高（三路门控需要调） | 低（结构简单） |
| 部署难度 | 高（实现复杂） | 低（容易集成） |

DeepSeek-V3.2 选 DSA 而不是 NSA，工程考量更多——DSA 的简洁让它在已有 MLA 推理框架上更容易集成。FlashMLA 库提供 sparse kernel，H800 上 prefill 640 TFlops / decode 410 TFlops。

实现细节我们不在这一节展开。核心记住：DSA = 轻量 indexer 打分 + top-N 选择 + 标准 attention，比 NSA 简洁、比 dense attention 高效。

## 6. Sparse Attention 的复杂度收益

把 dense attention、固定 pattern（sliding window）、learnable sparse（top-N）放在一起看复杂度。

对每个 query：
- Dense attention: 算 $N$ 个 key 的 attention，复杂度 $O(N d)$
- Sliding window: 算 $W$ 个邻近 key，复杂度 $O(W d)$
- Learnable sparse: 算 top-N 个 key + 打分开销，复杂度 $O(N d_{\text{small}} + K d)$，其中 $d_{\text{small}}$ 是 indexer 的低维

对长度 $N$ 的序列，全 attention 矩阵的复杂度：
- Dense: $O(N^2 d)$
- Sliding window: $O(N W d)$，$W$ 固定时是 $O(N d)$
- Learnable sparse: $O(N K d)$，$K$ 固定时也是 $O(N d)$

下面用代码具体算几个数。

In [ ]:
# Sparse attention 的复杂度对比

import matplotlib.pyplot as plt

d = 128           # head_dim
W = 1024          # sliding window size
K = 256           # sparse top-N

seq_lens = [4096, 8192, 16384, 32768, 65536, 131072, 262144]

dense_flops = [2 * N * N * d for N in seq_lens]
sliding_flops = [2 * N * W * d for N in seq_lens]
sparse_flops = [2 * N * K * d for N in seq_lens]

plt.figure(figsize=(11, 5))
plt.plot(seq_lens, dense_flops, 'o-', label=f'Dense: O(N²d)', linewidth=2, color='#e74c3c')
plt.plot(seq_lens, sliding_flops, 's-', label=f'Sliding Window (W={W}): O(N·W·d)', linewidth=2, color='#3498db')
plt.plot(seq_lens, sparse_flops, '^-', label=f'Sparse top-{K}: O(N·K·d)', linewidth=2, color='#2ecc71')
plt.xlabel('Sequence length N')
plt.ylabel('Attention FLOPs')
plt.title('Dense vs Sliding Window vs Sparse Attention 复杂度对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log', base=2)
plt.yscale('log')
plt.xticks(seq_lens, [f'{N//1024}K' for N in seq_lens])
plt.tight_layout()
plt.show()

# 在 128K 序列上的具体数字
N = 131072
print(f"序列长度 = {N//1024}K，head_dim = {d}")
print(f"  Dense attention:        {2*N*N*d:>15,} FLOPs")
print(f"  Sliding Window (W={W}): {2*N*W*d:>15,} FLOPs  ({2*N*N*d / (2*N*W*d):.0f}× 压缩)")
print(f"  Sparse top-{K}:         {2*N*K*d:>15,} FLOPs  ({2*N*N*d / (2*N*K*d):.0f}× 压缩)")
print()
print("关键观察：N 越大，sparse/sliding 相对 dense 的节省越显著")
print(f"在 128K 序列上，sparse top-{K} 比 dense 省 {2*N*N*d / (2*N*K*d):.0f}× 计算")

## 7. 端到端可训练：NSA 的关键工程贡献

讲一下为什么 NSA 强调「end-to-end 可训练」这件事。

早期的 sparse attention 工作里，top-K 选择是**不可微**的——你用 `torch.topk` 选出 key 的索引，索引本身是个离散操作，反向传播传不回去。这导致「应该选哪些 key」这件事没法用梯度训练，只能：
1. 用 RL（强化学习）训练选择策略
2. 用 Gumbel-softmax 等梯度估计
3. 用启发式规则固定选择策略

NSA 的关键贡献是把选择做成**可微**的。具体来说，压缩分支输出的 attention 分数被用作「软选择权重」——不是硬选 top-N，而是用 attention 分数加权所有 token（但分数本身经过了稀疏化处理）。这让整个流程可以用标准反向传播训练。

DSA 用了类似的策略——indexer 输出的 score 是连续可微的，top-N 选择用 straight-through estimator（前向 hard top-N，反向用软梯度）。

工程上这意味着 NSA 和 DSA 可以**直接替换 dense attention**，模型结构其他部分不变，从预训练阶段就接入。不需要先训 dense 模型再做转换。这是它们能进 DeepSeek-V3.2 这种生产模型的原因。

## 小结

- [ ] Softmax attention 后大部分权重接近 0——少数 key 拿走大部分权重，这是 sparse attention 的物理基础
- [ ] 固定 pattern（sliding window、local+global、block-sparse）简单但不灵活，无法适应「重要的 key 在远处」的场景
- [ ] Learnable sparse attention 用轻量 router 给 key 打分，选 top-N，只对选出的 key 算完整 attention
- [ ] NSA（DeepSeek 2025）三路并行：压缩（粗 attention）+ 选择（top-N 细 attention）+ 滑窗（局部），门控融合
- [ ] DSA（DeepSeek V3.2）是 NSA 的简化版：单 indexer 打分 + top-N + 标准 attention，部署更友好
- [ ] NSA/DSA 把 top-N 选择做成可微，整个流程 end-to-end 可训练，可直接替换 dense attention
- [ ] 复杂度：dense $O(N^2 d)$ → sparse/sliding $O(N K d)$，$K$ 固定时随 $N$ 线性增长

参考：[Native Sparse Attention (NSA)](https://arxiv.org/abs/2502.11089)、[DeepSeek-V3.2-Exp（含 DSA）](https://github.com/deepseek-ai/DeepSeek-V3.2-Exp)、[Longformer](https://arxiv.org/abs/2004.05150)、[Big Bird](https://arxiv.org/abs/2007.14062)。

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：Sparse attention 的复杂度节省**

给定 $N = 65536$（64K 上下文），$K = 512$（sparse top-N），$d = 128$。算 dense attention 和 sparse attention 的 FLOPs 比值。

小提示：dense $\approx 2 N^2 d$，sparse $\approx 2 N K d$。比值就是 $N/K$。

In [ ]:
# 作业 1：Sparse attention 复杂度节省

N = 65536
K = 512
d = 128

# TODO: 填入计算
dense_flops = None     # 2 * N * N * d
sparse_flops = None    # 2 * N * K * d
speedup = None         # dense_flops / sparse_flops

assert dense_flops is not None
assert sparse_flops is not None
assert speedup is not None

expected_dense = 2 * N * N * d
expected_sparse = 2 * N * K * d
expected_speedup = expected_dense / expected_sparse

assert dense_flops == expected_dense
assert sparse_flops == expected_sparse
assert abs(speedup - expected_speedup) < 0.001

print(f"✅ 作业 1 通过")
print(f"   Dense attention:  {dense_flops:,} FLOPs ({dense_flops:.2e})")
print(f"   Sparse attention: {sparse_flops:,} FLOPs ({sparse_flops:.2e})")
print(f"   加速比: {speedup:.0f}×")
print(f"   关键观察：sparse 让 64K 上下文的 attention 算力降到 dense 的 1/{int(speedup)}")

**作业 2：实现 Sliding Window Attention 的 mask**

构造一个长度 $N = 8$、窗口 $W = 4$ 的 causal + sliding window mask。mask 是 $N \times N$ 矩阵，允许注意的位置填 0，禁止的位置填 `-inf`。

小提示：先用 `torch.triu` 构造 causal mask，再用 `(row - col) > W - 1` 找出窗口外的位置，合在一起。

In [ ]:
# 作业 2：实现 sliding window mask

import torch

def make_sliding_window_mask(N, W):
    """构造 causal + sliding window mask

    返回 [N, N] 张量，0 = 允许注意，-inf = 禁止
    """
    # TODO: 补全这个函数
    # 1. causal mask: 上三角（不包括对角线）填 -inf
    # 2. sliding window: (row - col) > W - 1 的位置填 -inf
    pass


# 验证
mask = make_sliding_window_mask(N=8, W=4)

# 检查基本属性
assert mask.shape == (8, 8), f"shape 应为 (8, 8)，实际 {mask.shape}"
# (0, 0) 应该可见（自己）
assert mask[0, 0] == 0, "(0, 0) 应该可见"
# (3, 0) 应该不可见（距离 3，但 W=4，应该可见）—— 等等，距离 = 3，W=4 表示窗口包含 W 个 token，最大距离 W-1=3，所以 (3,0) 应该可见
assert mask[3, 0] == 0, "(3, 0) 距离 3，窗口 W=4 内，应可见"
# (4, 0) 应该不可见（距离 4，超出 W-1=3）
assert mask[4, 0] == float('-inf'), "(4, 0) 距离 4，超出窗口 W=4，应不可见"
# (i, j) 其中 j > i 应该 -inf（causal）
assert mask[0, 1] == float('-inf'), "(0, 1) 因果禁止，应不可见"

# 可视化
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 5))
plt.imshow(mask, cmap='RdYlGn', vmin=-10, vmax=0)
plt.colorbar(label='mask value')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.title(f'Sliding Window Mask (N=8, W=4)\n绿色=可见，红色=禁止')
plt.tight_layout()
plt.show()

print(f"✅ 作业 2 通过")
print(f"   Sliding window mask 构造正确")

**作业 3：top-N 选择的稀疏度计算**

给定一个 attention 矩阵 $N \times N$（N=64），每个 query 选 top-K=8 个 key。算 sparse attention 的稀疏度（被算的 entry 数 / 总 entry 数）。

小提示：每个 query 算 K 个 entry，共 $N$ 个 query，总 entry 数 = $N \times K$。原 attention 矩阵 entry 数 = $N^2$。稀疏度 = $NK / N^2 = K/N$。

In [ ]:
# 作业 3：top-N 稀疏度计算

N = 64
K = 8

# TODO: 填入计算
total_entries = None    # N * N（attention 矩阵总 entry 数）
sparse_entries = None   # N * K（实际算的 entry 数）
sparsity = None         # sparse_entries / total_entries（越小越省）

assert total_entries is not None
assert sparse_entries is not None
assert sparsity is not None

expected_total = N * N
expected_sparse = N * K
expected_sparsity = expected_sparse / expected_total

assert total_entries == expected_total
assert sparse_entries == expected_sparse
assert abs(sparsity - expected_sparsity) < 0.001

print(f"✅ 作业 3 通过")
print(f"   Attention 矩阵大小: {N}×{N} = {total_entries:,} entries")
print(f"   Sparse 实际计算: {N}×{K} = {sparse_entries:,} entries")
print(f"   稀疏度: {sparsity*100:.1f}%（只算 {sparsity*100:.0f}% 的 entry）")
print(f"   关键观察：稀疏度 = K/N = {K}/{N} = {K/N:.2f}")